# 00b — Saraga Hindustani: Data Preparation & Transcription

**Thesis context:** This notebook processes the Hindustani subset of the Saraga 1.5 dataset
(Srinivasamurthy et al., 2021). Recordings are full concert MP3s; we transcribe them to MIDI
using Basic-Pitch (Bitteur et al., 2022) to obtain symbolic representations suitable for
tokenisation.

**Key decisions:**
- **Backend:** Basic-Pitch is called with the explicit TF SavedModel path (`ICASSP_2022_MODEL_PATH`)
  to avoid the macOS CoreML floating-point exception that occurs when the CoreML backend is
  selected implicitly.
- **Metadata:** Per-track `.json` files provide raga label (`raags[0]['common_name']`), taal,
  and form. Where the JSON `raags` array is empty, the raga name is inferred from the concert
  folder name (e.g. "Raag Shree by Artist" → "Shree"). This fallback is flagged in the output.
- **File naming quirk:** Hindustani audio files carry a double `.mp3.mp3` extension (download
  artefact). The code strips the extra extension before constructing output paths.
- **Track selection:** 60 tracks are selected from the 108 available, stratified across ragas to
  maximise diversity.

**Output:**
- `data/processed/hindustani/midi/` — 60 transcribed MIDI files
- `data/metadata/hindustani_tracks.csv` — per-track metadata with raga labels and metadata source

**Time estimate:** ~5–8 min per file on CPU × 60 files ≈ **5–8 hours total**.
Run this notebook overnight or in a background terminal.


In [1]:
# IMPORTANT: Import TensorFlow before basic_pitch to ensure the TF SavedModel
# backend is loaded rather than CoreML (macOS). This must be the first import.
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

2026-06-21 20:10:25.569935: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/mohammadashraf/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow version: 2.12.0


In [2]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter was launched from.
# Searches upward for PROGRESS.md — the root marker.
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [3]:
import pandas as pd
import json
import shutil
import time

from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

print(f"Basic-Pitch model: {ICASSP_2022_MODEL_PATH}")

RAW_DIR   = PROJECT_ROOT / "datasets" / "indian_classical" / "saraga1.5_hindustani"
OUT_MIDI  = PROJECT_ROOT / "data" / "processed" / "hindustani" / "midi"
META_DIR  = PROJECT_ROOT / "data" / "metadata"

OUT_MIDI.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


Basic-Pitch model: /Users/mohammadashraf/Library/Python/3.9/lib/python/site-packages/basic_pitch/saved_models/icassp_2022/nmp


## 1. Scan all tracks and load metadata

In [4]:
# Load the file_paths.csv — the curated track list provided by the Saraga dataset.
# filepath column: "{concert_folder}/{raga_folder}/{piece_name}" (no extension)
fpc = pd.read_csv(RAW_DIR / "file_paths.csv", index_col=0)
print(f"file_paths.csv entries: {len(fpc)}")
fpc.head()

file_paths.csv entries: 108


,filepath,mbid
0,Raag Shree by Deborshee Bhattacharya/Raag Shre...,b3a43a82-b3c3-49bf-bd3d-db561c1ec355
1,Raag Lagan Gandhar & Nirgun Bhajan by Kumar Ga...,cf1b51f3-aa25-4fb2-bf63-f0271448abf0
2,Raag Lagan Gandhar & Nirgun Bhajan by Kumar Ga...,34c7f5c5-cd81-4a53-ae65-90c97f9effd2
3,"Raag Kalyan, Bhimpalasi & Dhani by Satyasheel ...",a9431b48-880b-4dcb-96f5-6f2c11879fb1
4,"Raag Kalyan, Bhimpalasi & Dhani by Satyasheel ...",f49ac932-a7d3-4031-ae44-acf5ea47eca1


In [5]:
def find_audio_file(raw_dir, filepath_stem):
    """Find the audio file for a given stem path from file_paths.csv."""
    # Hindustani: double extension .mp3.mp3
    for ext in [".mp3.mp3", ".mp3"]:
        p = raw_dir / (filepath_stem + ext)
        if p.exists():
            return p
    return None


def parse_hindustani_json(json_path):
    """Extract raga, taal, form from a Saraga Hindustani JSON metadata file."""
    try:
        meta = json.load(open(json_path, encoding="utf-8"))
        raags  = meta.get("raags", [])
        taals  = meta.get("taals", [])
        forms  = meta.get("forms", [])
        return {
            "title"  : meta.get("title", ""),
            "mbid"   : meta.get("mbid", ""),
            "raga"   : raags[0]["common_name"] if raags else None,
            "taal"   : taals[0]["common_name"] if taals else None,
            "form"   : forms[0]["common_name"] if forms else None,
            "length_ms": meta.get("length", None),
        }
    except Exception as e:
        return {"error": str(e)}


records = []
for _, row in fpc.iterrows():
    fp_stem = row["filepath"]
    audio   = find_audio_file(RAW_DIR, fp_stem)
    json_p  = RAW_DIR / (fp_stem + ".json")

    meta = parse_hindustani_json(json_p) if json_p.exists() else {}

    # Fallback: infer raga from concert folder name ("Raag Shree by Artist" → "Shree")
    raga = meta.get("raga")
    meta_src = "json"
    if not raga:
        concert_folder = fp_stem.split("/")[0]
        # Strip "Raag " prefix and artist suffix
        raga_raw = concert_folder.split(" by ")[0].replace("Raag ", "").strip()
        raga = raga_raw if raga_raw else "Unknown"
        meta_src = "folder_name"

    records.append({
        "filepath_stem": fp_stem,
        "audio_path"   : str(audio) if audio else None,
        "json_exists"  : json_path.exists() if (json_path := json_p) else False,
        "title"        : meta.get("title", ""),
        "mbid"         : row.get("mbid", ""),
        "raga"         : raga,
        "taal"         : meta.get("taal", ""),
        "form"         : meta.get("form", ""),
        "length_ms"    : meta.get("length_ms", None),
        "metadata_source": meta_src,
    })

tracks_df = pd.DataFrame(records)
print(f"Total tracks found: {len(tracks_df)}")
print(f"\nMetadata source distribution:")
print(tracks_df["metadata_source"].value_counts().to_string())
print(f"\nTracks with audio file: {tracks_df['audio_path'].notna().sum()}")
print(f"Unique ragas           : {tracks_df['raga'].nunique()}")

Total tracks found: 108

Metadata source distribution:
metadata_source
json           93
folder_name    15

Tracks with audio file: 108
Unique ragas           : 68


In [6]:
# Show raga distribution across available tracks
print("Raga distribution (top 20):")
print(tracks_df["raga"].value_counts().head(20).to_string())

Raga distribution (top 20):
raga
Shree                     5
Bhairabi                  5
Jog                       3
Miya malhar               3
Todi                      3
Geetinandan : Part-3      3
Marwa                     3
Lalat                     3
Ahir bhairav              2
Poorva & Gaud Malhar      2
Ramdasi malhar            2
Bhoopali & Paraj          2
Kedar & Sohani            2
Hameer                    2
Malkauns                  2
Gauri                     2
Sawani & Bahar            2
Subha Chale Sham Dhale    2
Puriya & Bhimpalasi       2
Multani                   2


## 2. Select 60 tracks — stratified across ragas

We sample proportionally across ragas to represent the breadth of the Hindustani
tradition in the dataset. Tracks with JSON metadata are preferred; folder-name fallbacks
are used only to reach the target count.


In [7]:
N = 60

# Prefer JSON-sourced tracks
json_tracks = tracks_df[
    (tracks_df["metadata_source"] == "json") & tracks_df["audio_path"].notna()
].copy()

sampled = (
    json_tracks
    .groupby("raga", group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), max(1, round(N * len(x) / len(json_tracks)))),
        random_state=42
    ))
)

if len(sampled) > N:
    sampled = sampled.sample(N, random_state=42)
elif len(sampled) < N:
    remaining = tracks_df[
        ~tracks_df["filepath_stem"].isin(sampled["filepath_stem"]) &
        tracks_df["audio_path"].notna()
    ]
    topup = remaining.sample(min(N - len(sampled), len(remaining)), random_state=42)
    sampled = pd.concat([sampled, topup])

sampled = sampled.reset_index(drop=True)
print(f"Selected {len(sampled)} tracks")
print(f"\nRaga distribution in selection:")
print(sampled["raga"].value_counts().to_string())

Selected 60 tracks

Raga distribution in selection:
raga
Bhairabi                 3
Lalat                    2
Shree                    2
Marwa                    2
Todi                     2
Jog                      2
Gawti                    1
Yaman kalyan             1
Jait Kalyan              1
Bibhas                   1
Miya malhar              1
Bahar                    1
Dagori                   1
Puriya dhanashree        1
Komal rishabh asavari    1
Sohini                   1
Bilaskhani todi          1
Triveni gauri            1
Dhani                    1
Maajh khamaj             1
Mishra kalingada         1
Hindol Pancham           1
Shuddha kalyan           1
Bhimpalas                1
Khamaj                   1
Lagan Gandhar            1
Kalavati                 1
Rageshri                 1
Lalit pancham            1
Marubihag                1
Abhogi                   1
Mishra piloo             1
Bairagi                  1
Nat bhairav              1
Des                      

/var/folders/88/d6f4kns57fz2gzfzhd0dktfw0000gn/T/ipykernel_42818/732675621.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  json_tracks


In [8]:
# Save track selection to metadata CSV before starting transcription
# (transcription can be safely interrupted and resumed)
sampled.to_csv(META_DIR / "hindustani_tracks.csv", index=False)
print("Selection saved → data/metadata/hindustani_tracks.csv")

Selection saved → data/metadata/hindustani_tracks.csv


## 3. Basic-Pitch transcription

**Runtime warning:** Each full concert recording (~35–50 minutes of audio) takes ~5–8 minutes
on CPU. For 60 files this is approximately **5–8 hours**. The loop is idempotent — already
transcribed files are skipped, so it is safe to interrupt and re-run.

**Transcription note:** Basic-Pitch transcribes all audible pitched content from the full mix.
For Hindustani recordings this captures the lead vocal/instrument plus harmonium and tabla.
This is a known limitation of automatic transcription from mixed audio; it is documented as
a methodological consideration in the thesis (Chapter 3).


In [9]:
midi_paths = []
failed     = []

for i, row in sampled.iterrows():
    audio_path = Path(row["audio_path"])
    raga_safe  = row["raga"].replace(" ", "_").replace("/", "_")[:30]
    out_name   = f"hindustani_{i:03d}_{raga_safe}.mid"
    out_path   = OUT_MIDI / out_name

    # Skip if already transcribed
    if out_path.exists():
        print(f"[{i+1:3d}/{len(sampled)}] SKIP  {out_name}")
        midi_paths.append(str(out_path))
        continue

    print(f"[{i+1:3d}/{len(sampled)}] Transcribing: {audio_path.name[:60]} ...", end="", flush=True)
    t0 = time.time()
    try:
        _, midi_data, _ = predict(str(audio_path), ICASSP_2022_MODEL_PATH)
        midi_data.write(str(out_path))
        elapsed = time.time() - t0
        print(f" {elapsed/60:.1f} min")
        midi_paths.append(str(out_path))
    except Exception as e:
        print(f" FAILED: {e}")
        failed.append({"index": i, "audio": str(audio_path), "error": str(e)})
        midi_paths.append(None)

print(f"\n=== Transcription complete ===")
print(f"Succeeded : {sum(p is not None for p in midi_paths)}")
print(f"Failed    : {len(failed)}")
if failed:
    for f in failed:
        print(f"  [{f['index']}] {f['audio']}: {f['error']}")

[  1/60] Transcribing: Raag Gavti.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Raag Bhoopali & Gavti by Omkar Dadarkar/Raag Gavti/Raag Gavti.mp3.mp3...


2026-06-21 20:11:17.144447: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '5758' with dtype float and shape [36,1,256]
	 [[{{node 5758}}]]


 4.2 min
[  2/60] Transcribing: Raag Abhogi.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Raag Abhogi & Megh by Anol Chatterjee/Raag Abhogi/Raag Abhogi.mp3.mp3...


2026-06-21 20:15:24.338684: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '15592' with dtype float and shape [36,1,256]
	 [[{{node 15592}}]]


 5.5 min
[  3/60] Transcribing: Thumri in Piloo.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Subha Chale Sham Dhale by Ajoy Chakrabarty/Thumri in Piloo/Thumri in Piloo.mp3.mp3...


...

 5.0 min
[ 59/60] Transcribing: Kalavati.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Geetinandan : Part-3 by Ajoy Chakrabarty/Kalavati/Kalavati.mp3.mp3...


2026-06-22 00:12:26.713614: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '530698' with dtype float and shape [36,1,256]
	 [[{{node 530698}}]]


 0.3 min
[ 60/60] Transcribing: Malkauns.mp3.mp3 ...Predicting MIDI for /Users/mohammadashraf/Desktop/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Geetinandan : Part-3 by Ajoy Chakrabarty/Malkauns/Malkauns.mp3.mp3...


2026-06-22 00:12:44.465755: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor '537356' with dtype float and shape [36,1,256]
	 [[{{node 537356}}]]


 0.3 min

=== Transcription complete ===
Succeeded : 60
Failed    : 0


## 4. Update metadata and verify

In [10]:
sampled = sampled.copy()
sampled["midi_path"] = midi_paths
sampled.to_csv(META_DIR / "hindustani_tracks.csv", index=False)

# Summary
midi_files = list(OUT_MIDI.glob("*.mid"))
print(f"MIDI files in output dir          : {len(midi_files)}")
print(f"Tracks with MIDI path recorded    : {sampled['midi_path'].notna().sum()}")
print(f"Raga coverage                     : {sampled['raga'].nunique()} unique ragas")
print("\n✓ Hindustani preparation complete.")

MIDI files in output dir          : 60
Tracks with MIDI path recorded    : 60
Raga coverage                     : 53 unique ragas

✓ Hindustani preparation complete.
